In [1]:
!pip install rdflib

  Using cached rdflib-7.6.0-py3-none-any.whl.metadata (12 kB)
Using cached rdflib-7.6.0-py3-none-any.whl (615 kB)



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\lenovo\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


Open this file and check how many triples are there to see if it can run properly.

In [1]:
from rdflib import Graph

url= ("https://raw.githubusercontent.com/Timoseee/windows-cyber-threat-kg/main/windows-cyber-threat-kg.ttl")
g = Graph()
g.parse(url,format="turtle")

print(f"Loaded {len(g)} RDF triples.")

Loaded 78 RDF triples.


Query all the actors and techniques in the file using SQL.

In [3]:
query_actor_technique = """
PREFIX cti: <https://github.com/Timoseee/windows-cyber-threat-kg/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?actorLabel ?techniqueLabel
WHERE {
    ?actor cti:usesTechnique ?technique .
    ?actor rdfs:label ?actorLabel .
    ?technique rdfs:label ?techniqueLabel .
}
"""

for row in g.query(query_actor_technique):
    print(f"{row.actorLabel} uses {row.techniqueLabel}")

APT29 (G0016) uses Spearphishing Attachment (T1566.001)


Check all vulnerabilities as well as their corresponding versions and the severity of the risks.

In [4]:
query_vulnerability = """
PREFIX cti: <https://github.com/Timoseee/windows-cyber-threat-kg/ontology#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?cve ?productLabel ?weaknessLabel
WHERE {
    ?cve rdf:type cti:Vulnerability ;
         cti:affectsProduct ?product ;
         cti:hasWeakness ?weakness .

    ?product rdfs:label ?productLabel .
    ?weakness rdfs:label ?weaknessLabel .
}
"""

for row in g.query(query_vulnerability):
    cve_id = str(row.cve).split("#")[-1]
    print(f"{cve_id} affects {row.productLabel}; weakness: {row.weaknessLabel}")

CVE_2024_30051 affects Windows 11 Version 23H2; weakness: CWE-122: Heap-based Buffer Overflow


Query the available attack techniques and tactics available on Windows.

In [5]:
query_technique_context = """
PREFIX cti: <https://github.com/Timoseee/windows-cyber-threat-kg/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?techniqueLabel ?tacticLabel ?platformLabel
WHERE {
    ?technique cti:belongsToTactic ?tactic ;
               cti:applicableToPlatform ?platform .

    ?technique rdfs:label ?techniqueLabel .
    ?tactic rdfs:label ?tacticLabel .
    ?platform rdfs:label ?platformLabel .
}
"""

for row in g.query(query_technique_context):
    print(
        f"Technique: {row.techniqueLabel} | "
        f"Tactic: {row.tacticLabel} | "
        f"Platform: {row.platformLabel}"
    )

Technique: Spearphishing Attachment (T1566.001) | Tactic: Initial Access | Platform: Windows


Draw graph

In [9]:
%pip install networkx matplotlib

  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
Using cached networkx-3.7-py3-none-any.whl (2.1 MB)
Note: you may need to restart the kernel to use updated packages.


Generate knowledge graph image from the info in the file

In [10]:
from rdflib import Namespace, RDFS
import networkx as nx
import matplotlib.pyplot as plt

CTI = Namespace(
    "https://github.com/Timoseee/windows-cyber-threat-kg/ontology#"
)

relationship_labels = {
    CTI.usesTechnique: "uses technique",
    CTI.belongsToTactic: "belongs to tactic",
    CTI.applicableToPlatform: "applicable to",
    CTI.affectsProduct: "affects product",
    CTI.hasWeakness: "has weakness",
    CTI.belongsToPlatform: "belongs to platform",
}

def display_name(node):
    label = next(g.objects(node, RDFS.label), None)
    return str(label) if label else str(node).split("#")[-1]

kg_graph = nx.DiGraph()

for relationship, edge_label in relationship_labels.items():
    for subject, obj in g.subject_objects(relationship):
        source = display_name(subject)
        target = display_name(obj)
        kg_graph.add_edge(source, target, label=edge_label)

plt.figure(figsize=(14, 8))
positions = nx.spring_layout(kg_graph, seed=7)

nx.draw(
    kg_graph,
    positions,
    with_labels=True,
    node_color="#B9DDF5",
    node_size=3200,
    font_size=9,
    font_weight="bold",
    arrows=True,
    arrowsize=20,
    width=1.8,
)

edge_labels = nx.get_edge_attributes(kg_graph, "label")
nx.draw_networkx_edge_labels(
    kg_graph,
    positions,
    edge_labels=edge_labels,
    font_size=8,
)

plt.title("Windows-focused Cybersecurity Threat Intelligence Knowledge Graph")
plt.axis("off")
plt.show()